# TabDiff big candidate pools (sampling only, no training)
Samples 50x the submission size from the already trained TabDiff models, for the selection optimiser.

Inputs to attach: the new dataset **tabdiff_comp_v2** (code + data) and the **output of the earlier training notebook (Version #1)**, which holds `ckpt/`.
Accelerator **GPU T4 x2**, Internet on. Run with **Save & Run All (Commit)**.

In [ ]:
import glob, os, shutil, subprocess, time
src = os.path.dirname(os.path.dirname(glob.glob('/kaggle/input/**/TabDiff/comp/optimize_selection.py', recursive=True)[0]))
WORK = '/kaggle/working/TabDiff'
if not os.path.exists(WORK):
    shutil.copytree(src, WORK)
os.chdir(WORK)
CKPT = {n: glob.glob(f'/kaggle/input/**/ckpt/{n}/baseline/best_ema_model*.pt', recursive=True) for n in ['unsw', 'nslkdd']}
assert all(len(v) >= 1 for v in CKPT.values()), 'attach the training notebook output that contains ckpt/'
assert all(len({os.path.basename(p) for p in v}) == 1 for v in CKPT.values()), 'found different checkpoints'
CKPT = {n: sorted(v)[0] for n, v in CKPT.items()}
print(CKPT)

In [ ]:
!pip install -q sdmetrics prdc icecream tomli==2.0.1 tomli-w category_encoders pyod
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

In [ ]:
# Make the density-plot step harmless if this plotly/kaleido combo cannot export images
p = 'tabdiff/trainer.py'; s = open(p).read()
if 'Skipping density plots' not in s:
    print('WARNING: trainer.py is not the patched version')

In [ ]:
FACTOR = 50
os.makedirs('/kaggle/working/pools', exist_ok=True)
jobs = {'unsw': 'cuda:0', 'nslkdd': 'cuda:1'}
procs = {}
for name, dev in jobs.items():
    log = open(f'/kaggle/working/pools/sample_{name}.log', 'w')
    procs[name] = subprocess.Popen(
        ['python', 'main.py', '--dataname', name, '--mode', 'sample', '--no_wandb', '--device', dev,
         '--ckpt_path', CKPT[name], '--pool_factor', str(FACTOR), '--seed', '7',
         '--submission_path', f'/kaggle/working/pools/{name}_pool{FACTOR}x.csv'],
        stdout=log, stderr=subprocess.STDOUT, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
start = time.time()
while any(p.poll() is None for p in procs.values()):
    time.sleep(300)
    print(f'{(time.time() - start) / 60:.0f} min', {n: p.poll() for n, p in procs.items()}, flush=True)
print({n: p.returncode for n, p in procs.items()})
!tail -n 3 /kaggle/working/pools/sample_*.log

In [ ]:
# csv -> parquet (several times smaller to download), then drop the csv
import pandas as pd
for name in jobs:
    f = f'/kaggle/working/pools/{name}_pool{FACTOR}x.csv'
    df = pd.read_csv(f)
    df.to_parquet(f.replace('.csv', '.parquet'), index=False)
    os.remove(f)
    print(name, df.shape)
!ls -lh /kaggle/working/pools

In [ ]:
# keep only the pools in the output
shutil.rmtree('/kaggle/working/TabDiff', ignore_errors=True)
!ls -lh /kaggle/working